# 💪🧠 Week 1 Lab: Simulating EMG-Driven Point-to-Point Reaching Movements

**Course:** Machine Learning for Neuroscience  
**Duration:** ~90 minutes  
**Tools:** Python 3.x, NumPy, Matplotlib

---

## Scenario

You are modeling a **single-joint (elbow) point-to-point reaching movement** along a radial arc. The forearm rotates from 45° (Point A) to 90° (Point B), driven by an **antagonist muscle pair**:

- **Agonist (e.g., Biceps):** Generates the initial accelerating torque
- **Antagonist (e.g., Triceps):** Brakes the movement to stop at the target

Real movements produce a characteristic **triphasic EMG pattern**:
1. **AG1 burst:** Agonist fires to accelerate the limb
2. **ANT burst:** Antagonist fires to decelerate and brake
3. **AG2 burst:** Small agonist burst to correct overshoot

---

## Bloom's Taxonomy Roadmap

| Level | Section | What You'll Do |
|-------|---------|----------------|
| 🟢 **Remember** | Part 1 | Set up simulation parameters and coordinate geometry |
| 🟡 **Understand** | Part 2 | Build the triphasic EMG burst model |
| 🟠 **Apply** | Part 3 | Convert EMG to torque and simulate joint dynamics |
| 🟦 **Analyze** | Part 4 | Examine how EMG timing affects movement profiles |
| 🟣 **Evaluate** | Part 5 | Assess movement quality and biological plausibility |
| 🔴 **Create** | Part 6 | Design EMG patterns for different reaching speeds |

## Instructions

- Fill in only the sections marked `### YOUR CODE HERE ###`.
- Hints are in collapsible sections below each exercise.
- Solutions are in the separate **Solutions** notebook.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 12,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'lines.linewidth': 1.5})
print("Setup complete! NumPy version:", np.__version__)

---
## 🟢 Part 1: Remember — Simulation Parameters

### Exercise 1.1: Physical Parameters
Fill in the missing calculations.

In [ ]:
# Exercise 1.1: Physical and Simulation Parameters
dt = 0.001         # time step (1 ms)
T = 1.5            # total duration (seconds)

# 1a. Create a time vector from 0 to T with step dt
time = ### YOUR CODE HERE ###
n_samples = len(time)

L = 0.35           # forearm length (m)
m = 1.5            # forearm mass (kg)
B_damp = 0.5       # joint damping (N·m·s/rad)

# 1b. Compute moment of inertia: I = (1/3) * m * L^2
I_inertia = ### YOUR CODE HERE ###

# 1c. Convert angles from degrees to radians
theta_A = ### YOUR CODE HERE ###  # 45 degrees
theta_B = ### YOUR CODE HERE ###  # 90 degrees

# 1d. Movement amplitude in radians
movement_amplitude = ### YOUR CODE HERE ###

# Verification
print(f"Time vector: {n_samples} samples, {T} s duration")
print(f"Moment of inertia: I = {I_inertia:.4f} kg·m²")
print(f"Movement: {np.rad2deg(theta_A):.0f}° → {np.rad2deg(theta_B):.0f}° ({np.rad2deg(movement_amplitude):.0f}° arc)")
assert len(time) == 1500, "Time vector should have 1500 samples"
assert np.isclose(I_inertia, (1/3) * m * L**2), "Check moment of inertia"
assert np.isclose(movement_amplitude, np.deg2rad(45)), "Movement should be 45°"
print("\n✅ Exercise 1.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- `np.arange(0, T, dt)` for time vector
- `I = (1/3) * m * L**2`
- `np.deg2rad(45)` converts degrees to radians
- `movement_amplitude = theta_B - theta_A`
</details>

### Exercise 1.2: Visualize Reaching Geometry

In [ ]:
# Exercise 1.2: Reaching Geometry
fig, ax = plt.subplots(figsize=(6, 6))

# Compute hand positions at start and target
hand_A = ### YOUR CODE HERE ###  # np.array([L * np.cos(theta_A), L * np.sin(theta_A)])
hand_B = ### YOUR CODE HERE ###  # same pattern with theta_B

# Draw arc path (do not modify below)
arc_angles = np.linspace(theta_A, theta_B, 100)
ax.plot(L * np.cos(arc_angles), L * np.sin(arc_angles), '--', color='gray',
        alpha=0.5, linewidth=2, label='Movement arc')
ax.plot([0, hand_A[0]], [0, hand_A[1]], 'b-o', linewidth=3, markersize=8, label='Point A (start)')
ax.plot([0, hand_B[0]], [0, hand_B[1]], 'r-o', linewidth=3, markersize=8, label='Point B (target)')
ax.plot(0, 0, 'ko', markersize=12, zorder=5)
ax.annotate('Elbow', (0.01, -0.03), fontsize=11, fontweight='bold')
ax.annotate(f'A ({np.rad2deg(theta_A):.0f}°)', hand_A + 0.01, fontsize=11, color='blue')
ax.annotate(f'B ({np.rad2deg(theta_B):.0f}°)', hand_B + np.array([0.01, 0.01]), fontsize=11, color='red')
ax.set_xlim(-0.1, 0.45); ax.set_ylim(-0.05, 0.45); ax.set_aspect('equal')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('Point-to-Point Reaching: Radial Arc Movement')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`hand_A = np.array([L * np.cos(theta_A), L * np.sin(theta_A)])`
</details>

---
## 🟡 Part 2: Understand — The Triphasic EMG Model

Each EMG burst is modeled as a Gaussian envelope × rectified noise:
$$\text{EMG}(t) = A \cdot \exp\left(-\frac{(t - \mu)^2}{2\sigma^2}\right) \cdot |\text{noise}(t)|$$

### Exercise 2.1: Single EMG Burst

In [ ]:
# Exercise 2.1: Single EMG Burst
def emg_burst(time, amplitude, center, width):
    """Generate a single EMG burst as Gaussian envelope * rectified noise."""
    # Step 1: Gaussian envelope = amplitude * exp(-(time - center)^2 / (2*width^2))
    envelope = ### YOUR CODE HERE ###

    # Step 2: Rectified noise = absolute value of random normal samples
    noise = ### YOUR CODE HERE ###

    # Step 3: EMG signal = envelope * noise
    emg_signal = ### YOUR CODE HERE ###

    return envelope, emg_signal

np.random.seed(42)
env_test, emg_test = emg_burst(time, amplitude=1.0, center=0.1, width=0.03)
peak_idx = np.argmax(env_test)
print(f"Envelope peak at t = {time[peak_idx]*1000:.1f} ms (expected ~100 ms)")
assert np.isclose(time[peak_idx], 0.1, atol=0.002)
assert np.all(emg_test >= 0)
assert env_test.max() <= 1.01
print("\n✅ Exercise 2.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- Gaussian: `amplitude * np.exp(-((time - center)**2) / (2 * width**2))`
- Noise: `np.abs(np.random.randn(len(time)))`
- EMG: `envelope * noise`
</details>

### Exercise 2.2: Full Triphasic EMG

In [ ]:
# Exercise 2.2: Triphasic EMG Pattern
def generate_triphasic_emg(time, params):
    """Generate triphasic EMG for agonist-antagonist muscle pair."""
    np.random.seed(42)
    env_ag1, raw_ag1 = emg_burst(time, params['ag1_amp'], params['ag1_center'], params['ag1_width'])

    # Generate the ANT burst using emg_burst with ant parameters
    env_ant, raw_ant = ### YOUR CODE HERE ###

    # Generate the AG2 burst using emg_burst with ag2 parameters
    env_ag2, raw_ag2 = ### YOUR CODE HERE ###

    tonic_ag = params['tonic_agonist'] * np.abs(np.random.randn(len(time)))
    # Agonist EMG = AG1 + AG2 + tonic
    emg_agonist = ### YOUR CODE HERE ###

    tonic_ant = params['tonic_antagonist'] * np.abs(np.random.randn(len(time)))
    # Antagonist EMG = ANT + tonic
    emg_antagonist = ### YOUR CODE HERE ###

    envelopes = {
        'ag1': env_ag1, 'ant': env_ant, 'ag2': env_ag2,
        'agonist_total': env_ag1 + env_ag2 + params['tonic_agonist'],
        'antagonist_total': env_ant + params['tonic_antagonist']
    }
    return emg_agonist, emg_antagonist, envelopes

emg_params = {
    'ag1_amp': 1.3,   'ag1_center': 0.10,  'ag1_width': 0.04,
    'ant_amp': 0.85,  'ant_center': 0.20,  'ant_width': 0.04,
    'ag2_amp': 0.26,  'ag2_center': 0.30,  'ag2_width': 0.03,
    'tonic_agonist': 0.03, 'tonic_antagonist': 0.03,
}
emg_ag, emg_ant, envelopes = generate_triphasic_emg(time, emg_params)
print(f"Agonist EMG: mean={emg_ag.mean():.4f}, max={emg_ag.max():.3f}")
print(f"Antagonist EMG: mean={emg_ant.mean():.4f}, max={emg_ant.max():.3f}")
assert np.all(emg_ag >= 0); assert np.all(emg_ant >= 0)
assert envelopes['ag1'].max() > envelopes['ag2'].max()
print("\n✅ Exercise 2.2 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- ANT: `emg_burst(time, params['ant_amp'], params['ant_center'], params['ant_width'])`
- AG2: same pattern with `'ag2_...'` keys
- Agonist: `raw_ag1 + raw_ag2 + tonic_ag`
- Antagonist: `raw_ant + tonic_ant`
</details>

### Exercise 2.3: Visualize EMG

In [ ]:
# Exercise 2.3: Visualize EMG
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.fill_between(time, 0, emg_ag, alpha=0.3, color='steelblue', label='Raw EMG')
# Plot the agonist envelope
### YOUR CODE HERE ###  # ax1.plot(time, envelopes['agonist_total'], ...)
ax1.axvline(x=emg_params['ag1_center'], color='navy', linestyle=':', alpha=0.6, label='AG1 peak')
ax1.axvline(x=emg_params['ag2_center'], color='cornflowerblue', linestyle=':', alpha=0.6, label='AG2 peak')
ax1.set_ylabel('Agonist EMG (a.u.)'); ax1.set_title('Agonist (Biceps)')
ax1.legend(loc='upper right', fontsize=9); ax1.set_xlim(0, 0.6)

ax2.fill_between(time, 0, emg_ant, alpha=0.3, color='salmon', label='Raw EMG')
# Plot the antagonist envelope
### YOUR CODE HERE ###  # ax2.plot(time, envelopes['antagonist_total'], ...)
ax2.axvline(x=emg_params['ant_center'], color='darkred', linestyle=':', alpha=0.6, label='ANT peak')
ax2.set_ylabel('Antagonist EMG (a.u.)'); ax2.set_xlabel('Time (s)')
ax2.set_title('Antagonist (Triceps)'); ax2.legend(loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- `ax1.plot(time, envelopes['agonist_total'], 'b-', linewidth=2, label='Envelope')`
- `ax2.plot(time, envelopes['antagonist_total'], 'r-', linewidth=2, label='Envelope')`
</details>

---
## 🟠 Part 3: Apply — From EMG to Movement

Conversion pipeline: EMG envelope → Torque → Movement via Newton's 2nd law:
$$I \cdot \ddot{\theta} = \tau_{net} - B \cdot \dot{\theta}$$

### Exercise 3.1: EMG to Torque

In [ ]:
# Exercise 3.1: EMG to Torque
def emg_to_torque(emg_agonist_env, emg_antagonist_env, gain_ag=6.0, gain_ant=6.0):
    # Agonist: positive torque (drives movement)
    tau_agonist = ### YOUR CODE HERE ###
    # Antagonist: NEGATIVE torque (brakes movement)
    tau_antagonist = ### YOUR CODE HERE ###
    # Net torque
    tau_net = ### YOUR CODE HERE ###
    return tau_net, tau_agonist, tau_antagonist

tau_net, tau_ag, tau_ant = emg_to_torque(envelopes['agonist_total'], envelopes['antagonist_total'])
print(f"Peak agonist torque:     {tau_ag.max():.3f} N·m")
print(f"Peak antagonist torque:  {tau_ant.min():.3f} N·m")
assert tau_ag.max() > 0; assert tau_ant.min() < 0
print("\n✅ Exercise 3.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- `tau_agonist = gain_ag * emg_agonist_env`
- `tau_antagonist = -gain_ant * emg_antagonist_env` (note the minus!)
- `tau_net = tau_agonist + tau_antagonist`
</details>

### Exercise 3.2: Simulate Joint Dynamics (Euler Integration)

In [ ]:
# Exercise 3.2: Joint Dynamics (Euler Integration)
# Equation: I * alpha = tau_net - B * omega
def simulate_joint_dynamics(tau_net, dt, I, B, theta_init):
    n = len(tau_net)
    theta = np.zeros(n); omega = np.zeros(n); alpha = np.zeros(n)
    theta[0] = theta_init; omega[0] = 0.0
    for i in range(n - 1):
        # Angular acceleration: alpha = (tau_net - B*omega) / I
        alpha[i] = ### YOUR CODE HERE ###
        # Update velocity: omega(t+dt) = omega(t) + alpha(t) * dt
        omega[i + 1] = ### YOUR CODE HERE ###
        # Update angle: theta(t+dt) = theta(t) + omega(t) * dt
        theta[i + 1] = ### YOUR CODE HERE ###
    alpha[-1] = (tau_net[-1] - B * omega[-1]) / I
    return theta, omega, alpha

theta, omega, alpha_arr = simulate_joint_dynamics(tau_net, dt, I_inertia, B_damp, theta_A)
print(f"Final angle: {np.rad2deg(theta[-1]):.1f}° (target: {np.rad2deg(theta_B):.1f}°)")
print(f"Peak velocity: {np.rad2deg(np.max(np.abs(omega))):.1f} °/s")
assert theta[0] == theta_A; assert omega[0] == 0.0
print("\n✅ Exercise 3.2 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- `alpha[i] = (tau_net[i] - B * omega[i]) / I`
- `omega[i+1] = omega[i] + alpha[i] * dt`
- `theta[i+1] = theta[i] + omega[i] * dt`
</details>

### Exercise 3.3: Plot Complete Movement Profile

In [ ]:
# Exercise 3.3: Complete Movement Profile
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

# Panel 1: EMG (provided)
axes[0].fill_between(time, 0, emg_ag, alpha=0.25, color='steelblue')
axes[0].plot(time, envelopes['agonist_total'], 'b-', linewidth=2, label='Agonist')
axes[0].fill_between(time, 0, emg_ant, alpha=0.25, color='salmon')
axes[0].plot(time, envelopes['antagonist_total'], 'r-', linewidth=2, label='Antagonist')
axes[0].set_ylabel('EMG (a.u.)'); axes[0].set_title('EMG → Torque → Kinematics')
axes[0].legend(loc='upper right', fontsize=9)

# Panel 2: Plot agonist torque (blue), antagonist torque (red), net torque (black)
### YOUR CODE HERE ###
axes[1].axhline(y=0, color='gray', alpha=0.3); axes[1].set_ylabel('Torque (N·m)')
axes[1].legend(loc='upper right', fontsize=9)

# Panel 3: Plot angular velocity in degrees/s (green)
### YOUR CODE HERE ###
axes[2].axhline(y=0, color='gray', alpha=0.3); axes[2].set_ylabel('Velocity (°/s)')

# Panel 4: Plot joint angle in degrees (black)
### YOUR CODE HERE ###
axes[3].axhline(y=np.rad2deg(theta_B), color='red', linestyle='--', alpha=0.5, label='Target')
axes[3].axhline(y=np.rad2deg(theta_A), color='blue', linestyle='--', alpha=0.5, label='Start')
axes[3].set_ylabel('Angle (°)'); axes[3].set_xlabel('Time (s)'); axes[3].legend(loc='right', fontsize=9)

for ax in axes: ax.set_xlim(0, 0.6)
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Panel 2: `axes[1].plot(time, tau_ag, 'b-', ...)`, `axes[1].plot(time, tau_ant, 'r-', ...)`, `axes[1].plot(time, tau_net, 'k-', ...)`
Panel 3: `axes[2].plot(time, np.rad2deg(omega), 'g-', ...)`
Panel 4: `axes[3].plot(time, np.rad2deg(theta), 'k-', ...)`
</details>

---
## 🟦 Part 4: Analyze — Effect of Antagonist Timing

### Exercise 4.1: Vary Antagonist Timing
This code is complete — run it and study the output.

In [ ]:
# Exercise 4.1: Antagonist Timing Analysis
ant_timings = [0.12, 0.16, 0.20, 0.24, 0.28]
timing_labels = ['Very early', 'Early', 'Normal', 'Late', 'Very late']
colors_timing = ['#2ecc71', '#27ae60', '#2980b9', '#e74c3c', '#c0392b']

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
results = []
for i, ant_time in enumerate(ant_timings):
    params_mod = emg_params.copy()
    params_mod['ant_center'] = ant_time
    _, _, env_mod = generate_triphasic_emg(time, params_mod)
    tau_mod, _, _ = emg_to_torque(env_mod['agonist_total'], env_mod['antagonist_total'])
    theta_mod, omega_mod, _ = simulate_joint_dynamics(tau_mod, dt, I_inertia, B_damp, theta_A)
    overshoot = np.rad2deg(np.max(theta_mod)) - np.rad2deg(theta_mod[-1])
    results.append({'label': timing_labels[i], 'ant_ms': ant_time*1000,
                    'final': np.rad2deg(theta_mod[-1]),
                    'peak_v': np.rad2deg(np.max(omega_mod)), 'overshoot': overshoot})
    axes[0].plot(time, np.rad2deg(theta_mod), color=colors_timing[i], linewidth=2,
                 label=f'{timing_labels[i]} ({ant_time*1000:.0f} ms)')
    axes[1].plot(time, np.rad2deg(omega_mod), color=colors_timing[i], linewidth=2)

axes[0].axhline(y=np.rad2deg(theta_B), color='black', linestyle='--', alpha=0.5, label='Target')
axes[0].set_ylabel('Joint Angle (°)'); axes[0].set_title('Effect of Antagonist Timing')
axes[0].legend(loc='right', fontsize=8)
axes[1].axhline(y=0, color='gray', alpha=0.3); axes[1].set_ylabel('Velocity (°/s)')
axes[1].set_xlabel('Time (s)')
for ax in axes: ax.set_xlim(0, 0.8)
plt.tight_layout(); plt.show()

print(f"{'Timing':<14} {'ANT(ms)':>8} {'Final(°)':>9} {'PeakV(°/s)':>11} {'Overshoot(°)':>13}")
print("-" * 58)
for r in results:
    print(f"{r['label']:<14} {r['ant_ms']:>8.0f} {r['final']:>9.1f} {r['peak_v']:>11.1f} {r['overshoot']:>13.1f}")

### Exercise 4.2: Interpretation Questions

Double-click this cell to edit.

1. How does peak velocity change with antagonist timing? Why?
2. Which timing conditions produce overshoot? Why is this significant?
3. How does this relate to the role of the cerebellum in movement timing?

**Your Answers:**

1. _[Your answer]_
2. _[Your answer]_
3. _[Your answer]_

---
## 🟣 Part 5: Evaluate — Movement Quality

### Exercise 5.1: Compute Quality Metrics

In [ ]:
# Exercise 5.1: Movement Quality Metrics
def movement_quality(theta, omega, time, theta_target, vel_threshold_deg=5.0):
    omega_deg = np.rad2deg(omega)
    # 1. Endpoint error (degrees)
    endpoint_error = ### YOUR CODE HERE ###
    # 2. Movement onset/offset
    moving = np.abs(omega_deg) > vel_threshold_deg
    if np.any(moving):
        onset_idx = np.argmax(moving)
        offset_idx = len(moving) - 1 - np.argmax(moving[::-1])
        movement_time = time[offset_idx] - time[onset_idx]
    else:
        onset_idx, offset_idx, movement_time = 0, len(time) - 1, 0.0
    # 3. Peak velocity index
    peak_vel_idx = ### YOUR CODE HERE ###  # index of max absolute velocity
    peak_velocity = omega_deg[peak_vel_idx]
    # 4. Symmetry ratio
    if movement_time > 0:
        time_to_peak = time[peak_vel_idx] - time[onset_idx]
        symmetry_ratio = ### YOUR CODE HERE ###
    else:
        symmetry_ratio = 0.0
    # 5. Velocity peaks (provided)
    if offset_idx > onset_idx:
        omega_movement = omega_deg[onset_idx:offset_idx + 1]
        if len(omega_movement) > 2:
            vel_deriv = np.diff(omega_movement)
            sign_changes = np.sum(np.diff(np.sign(vel_deriv)) != 0)
            n_velocity_peaks = max(1, (sign_changes + 1) // 2)
        else: n_velocity_peaks = 1
    else: n_velocity_peaks = 1
    return {'endpoint_error_deg': endpoint_error, 'movement_time_ms': movement_time * 1000,
            'peak_velocity_deg_s': peak_velocity, 'symmetry_ratio': symmetry_ratio,
            'n_velocity_peaks': n_velocity_peaks}

metrics = movement_quality(theta, omega, time, theta_B)
print("Movement Quality Metrics")
for k, v in metrics.items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")
assert metrics['endpoint_error_deg'] >= 0
assert metrics['movement_time_ms'] > 0
assert 0 < metrics['symmetry_ratio'] < 1
print("\n✅ Exercise 5.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- `endpoint_error = np.abs(np.rad2deg(theta[-1]) - np.rad2deg(theta_target))`
- `peak_vel_idx = np.argmax(np.abs(omega_deg))`
- `symmetry_ratio = time_to_peak / movement_time`
</details>

### Exercise 5.2: Compare Quality Across Timings
Run this cell (no code to fill in) and answer the questions below.

In [ ]:
# Solution 5.2: Quality Comparison Across Timings
all_metrics = []
for i, ant_time in enumerate(ant_timings):
    pm = emg_params.copy(); pm['ant_center'] = ant_time
    _, _, env_m = generate_triphasic_emg(time, pm)
    tau_m, _, _ = emg_to_torque(env_m['agonist_total'], env_m['antagonist_total'])
    th_m, om_m, _ = simulate_joint_dynamics(tau_m, dt, I_inertia, B_damp, theta_A)
    mq = movement_quality(th_m, om_m, time, theta_B)
    mq['label'] = timing_labels[i]; mq['ant_ms'] = ant_time * 1000
    all_metrics.append(mq)

print(f"{'Timing':<14} {'ANT(ms)':>8} {'Error°':>8} {'MT(ms)':>8} {'PeakV':>8} {'Sym':>6} {'Peaks':>6}")
print("-" * 56)
for m in all_metrics:
    print(f"{m['label']:<14} {m['ant_ms']:>8.0f} {m['endpoint_error_deg']:>8.2f} "
          f"{m['movement_time_ms']:>8.0f} {m['peak_velocity_deg_s']:>8.1f} "
          f"{m['symmetry_ratio']:>6.3f} {m['n_velocity_peaks']:>6}")
print("\n✅ Exercise 5.2 passed!")

### Exercise 5.3: Evaluation Questions

1. Which timing produces the best movement quality overall?
2. How does the symmetry ratio relate to the minimum jerk model prediction of 0.5?
3. In cerebellar patients, which aspect of the EMG pattern is likely disrupted?

**Your Answers:**

1. _[Your answer]_
2. _[Your answer]_
3. _[Your answer]_

---
## 🔴 Part 6: Create — Speed-Accuracy Tradeoff

### Exercise 6.1: Design EMG for Slow and Fast Reaching
Adjust burst amplitudes, widths, and centers to achieve different movement speeds.

In [ ]:
# Exercise 6.1: Design EMG for different speeds
# SLOW: lower amplitude, wider bursts, spread timing
params_slow = {
    'ag1_amp': ### YOUR CODE HERE ###,  'ag1_center': ### YOUR CODE HERE ###,  'ag1_width': ### YOUR CODE HERE ###,
    'ant_amp': ### YOUR CODE HERE ###,  'ant_center': ### YOUR CODE HERE ###,  'ant_width': ### YOUR CODE HERE ###,
    'ag2_amp': ### YOUR CODE HERE ###,  'ag2_center': ### YOUR CODE HERE ###,  'ag2_width': ### YOUR CODE HERE ###,
    'tonic_agonist': 0.03, 'tonic_antagonist': 0.03,
}
params_normal = emg_params.copy()
# FAST: higher amplitude, narrower bursts, compressed timing
params_fast = {
    'ag1_amp': ### YOUR CODE HERE ###,  'ag1_center': ### YOUR CODE HERE ###,  'ag1_width': ### YOUR CODE HERE ###,
    'ant_amp': ### YOUR CODE HERE ###,  'ant_center': ### YOUR CODE HERE ###,  'ant_width': ### YOUR CODE HERE ###,
    'ag2_amp': ### YOUR CODE HERE ###,  'ag2_center': ### YOUR CODE HERE ###,  'ag2_width': ### YOUR CODE HERE ###,
    'tonic_agonist': 0.03, 'tonic_antagonist': 0.03,
}

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for label, params, color in [('Slow', params_slow, '#2ecc71'),
                              ('Normal', params_normal, '#3498db'),
                              ('Fast', params_fast, '#e74c3c')]:
    e_ag, e_ant, env = generate_triphasic_emg(time, params)
    tn, _, _ = emg_to_torque(env['agonist_total'], env['antagonist_total'])
    th, om, _ = simulate_joint_dynamics(tn, dt, I_inertia, B_damp, theta_A)
    mq = movement_quality(th, om, time, theta_B)
    print(f"{label:>7}: Final={np.rad2deg(th[-1]):.1f}°, MT={mq['movement_time_ms']:.0f}ms, "
          f"Error={mq['endpoint_error_deg']:.1f}°")
    axes[0].plot(time, env['agonist_total'], color=color, linewidth=2, label=f'{label} (AG)')
    axes[0].plot(time, env['antagonist_total'], color=color, linewidth=2, linestyle='--')
    axes[1].plot(time, np.rad2deg(om), color=color, linewidth=2, label=label)
    axes[2].plot(time, np.rad2deg(th), color=color, linewidth=2,
                 label=f'{label}: err={mq["endpoint_error_deg"]:.1f}°, MT={mq["movement_time_ms"]:.0f}ms')
axes[0].set_ylabel('EMG Envelope'); axes[0].legend(fontsize=8)
axes[0].set_title('Speed-Accuracy Tradeoff')
axes[1].set_ylabel('Velocity (°/s)'); axes[1].axhline(y=0, color='gray', alpha=0.3); axes[1].legend(fontsize=9)
axes[2].set_ylabel('Angle (°)'); axes[2].set_xlabel('Time (s)')
axes[2].axhline(y=np.rad2deg(theta_B), color='black', linestyle='--', alpha=0.5, label='Target')
axes[2].legend(fontsize=8, loc='right')
for ax in axes: ax.set_xlim(0, 0.8)
plt.tight_layout(); plt.show()

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

**Slow** — lower amplitudes (~0.9), wider widths (~0.06), spread centers (~0.15, 0.32, 0.45):
```python
params_slow = {'ag1_amp': 0.9, 'ag1_center': 0.15, 'ag1_width': 0.06,
  'ant_amp': 0.59, 'ant_center': 0.32, 'ant_width': 0.06,
  'ag2_amp': 0.14, 'ag2_center': 0.45, 'ag2_width': 0.04, ...}
```
**Fast** — higher amplitudes (~3.5), narrow widths (~0.025), compressed centers (~0.06, 0.12, 0.18):
```python
params_fast = {'ag1_amp': 3.5, 'ag1_center': 0.06, 'ag1_width': 0.025,
  'ant_amp': 2.80, 'ant_center': 0.12, 'ant_width': 0.025,
  'ag2_amp': 0.53, 'ag2_center': 0.18, 'ag2_width': 0.02, ...}
```
</details>

### Exercise 6.2: Synthesis Questions

1. What parameters did you change for faster movements? How does this relate to Fitts' Law?
2. Did fast movements show overshoot? How did you compensate?
3. How would you extend this to a two-joint (shoulder + elbow) reaching model?

**Your Answers:**

1. _[Your answer]_
2. _[Your answer]_
3. _[Your answer]_

---
## 🎯 Lab Summary

| Bloom's Level | What You Accomplished |
|---|---|
| 🟢 **Remember** | Physical parameters, coordinate geometry |
| 🟡 **Understand** | Gaussian EMG bursts and triphasic activation model |
| 🟠 **Apply** | EMG → torque → kinematics via Euler integration |
| 🟦 **Analyze** | Antagonist timing effects on velocity and overshoot |
| 🟣 **Evaluate** | Movement quality metrics and clinical connections |
| 🔴 **Create** | EMG strategies for different movement speeds |